# Scraping daftar universitas dari US News

Target:

- https://www.usnews.com/best-graduate-schools/top-computer-science-schools/computer-science-rankings

Field yang diambil:

- `rank`
- `rank_label`
- `school_name`
- `city`
- `state`
- `tuition`
- `enrollment_full_time`
- `summary`
- `url`

Output:

- `output/usnews_cs_raw.html`
- `output/usnews_cs_rankings.csv`
- `output/usnews_cs_rankings.xlsx`

Catatan:

- Secara default notebook ini mengambil ulang halaman live agar tidak berhenti di 10 hasil pertama.
- Jika ingin parse file cache lama, set `USE_CACHE_IF_AVAILABLE = True`.
- `Selenium` diprioritaskan agar tombol `Load More` bisa diklik beberapa kali.

In [ ]:
%pip install -q requests beautifulsoup4 pandas curl-cffi selenium webdriver-manager openpyxl

In [2]:
from __future__ import annotations

import re
import time
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

URL = "https://www.usnews.com/best-graduate-schools/top-computer-science-schools/computer-science-rankings"
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

RAW_HTML_PATH = OUTPUT_DIR / "usnews_cs_raw.html"
CSV_PATH = OUTPUT_DIR / "usnews_cs_rankings.csv"
XLSX_PATH = OUTPUT_DIR / "usnews_cs_rankings.xlsx"

USE_CACHE_IF_AVAILABLE = False
MAX_LOAD_MORE_CLICKS = 25
FETCH_PRIORITY = ["selenium", "curl_cffi", "requests"]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Cache-Control": "no-cache",
    "Pragma": "no-cache",
}

STATE_CODES = {
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA", "HI", "ID", "IL", "IN", "IA",
    "KS", "KY", "LA", "ME", "MD", "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ",
    "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC", "SD", "TN", "TX", "UT", "VT",
    "VA", "WA", "WV", "WI", "WY", "DC",
}


def clean_text(value):
    if value is None:
        return None
    text = " ".join(str(value).split())
    text = text.replace("\u00c2\u00bb", "\u00bb").replace("\u00c2", "")
    text = text.replace("\u0165", "\u00bb")
    return text.strip() or None


def normalize_space_around_comma(text):
    text = clean_text(text)
    if not text:
        return None
    return re.sub(r"\s*,\s*", ", ", text)


def fetch_with_curl_cffi(url):
    try:
        from curl_cffi import requests as curl_requests
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError("Modul 'curl_cffi' belum terpasang.") from exc

    last_error = None
    for attempt in range(1, 4):
        try:
            response = curl_requests.get(
                url,
                headers=HEADERS,
                impersonate="chrome124",
                timeout=90,
            )
            response.raise_for_status()
            if response.text and len(response.text) > 5000:
                return response.text
            raise RuntimeError("HTML terlalu pendek, kemungkinan masih diblokir")
        except Exception as exc:
            last_error = exc
            time.sleep(attempt * 2)
    raise last_error


def fetch_with_requests(url):
    session = requests.Session()
    last_error = None
    for attempt in range(1, 4):
        try:
            response = session.get(url, headers=HEADERS, timeout=(20, 90))
            response.raise_for_status()
            if response.text and len(response.text) > 5000:
                return response.text
            raise RuntimeError("HTML terlalu pendek, kemungkinan masih diblokir")
        except Exception as exc:
            last_error = exc
            time.sleep(attempt * 2)
    raise last_error


def fetch_with_selenium(url):
    try:
        from selenium import webdriver
        from selenium.webdriver.chrome.options import Options
        from selenium.webdriver.chrome.service import Service
        from selenium.webdriver.common.by import By
        from selenium.webdriver.support import expected_conditions as EC
        from selenium.webdriver.support.ui import WebDriverWait
        from webdriver_manager.chrome import ChromeDriverManager
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError("Selenium belum terpasang. Jalankan cell install dependency terlebih dulu.") from exc

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1600,5000")
    options.add_argument(f"user-agent={HEADERS['User-Agent']}")

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    try:
        driver.get(url)
        WebDriverWait(driver, 30).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        time.sleep(5)

        for _ in range(MAX_LOAD_MORE_CLICKS):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
            try:
                button = WebDriverWait(driver, 3).until(EC.element_to_be_clickable((By.ID, "load-more-button")))
            except Exception:
                break
            if not button.is_displayed():
                break
            driver.execute_script("arguments[0].click();", button)
            time.sleep(4)

        html = driver.page_source
        if not html or len(html) < 5000:
            raise RuntimeError("HTML dari Selenium terlalu pendek")
        return html
    finally:
        driver.quit()


def fetch_html(url):
    if USE_CACHE_IF_AVAILABLE and RAW_HTML_PATH.exists():
        print(f"Menggunakan cache HTML: {RAW_HTML_PATH.resolve()}")
        return RAW_HTML_PATH.read_text(encoding="utf-8")

    fetchers = {
        "selenium": fetch_with_selenium,
        "curl_cffi": fetch_with_curl_cffi,
        "requests": fetch_with_requests,
    }
    errors = []
    for label in FETCH_PRIORITY:
        try:
            html = fetchers[label](url)
            print(f"HTML berhasil diambil dengan {label}")
            RAW_HTML_PATH.write_text(html, encoding="utf-8")
            return html
        except Exception as exc:
            print(f"{label} gagal: {exc}")
            errors.append(f"{label}: {exc}")

    raise RuntimeError("Tidak bisa mengambil halaman target. Detail error: " + " | ".join(errors))


def extract_location(card):
    for p in card.select("p"):
        text = normalize_space_around_comma(" ".join(p.stripped_strings))
        if not text:
            continue
        if "Read More" in text:
            continue
        if "TUITION AND FEES" in text or "ENROLLMENT" in text:
            continue
        if "$" in text:
            continue
        return text
    return None


def split_city_state(location):
    if not location:
        return None, None
    if "," in location:
        city, state = [item.strip() for item in location.rsplit(",", 1)]
        return city or None, state or None
    if location in STATE_CODES:
        return None, location
    return location, None


def extract_rank_info(card):
    rank_anchor = card.select_one(".rank-list-item a")
    if not rank_anchor:
        return None, None
    rank_label = clean_text(" ".join(rank_anchor.stripped_strings))
    match = re.search(r"#\s*(\d+)", rank_label or "")
    rank = int(match.group(1)) if match else None
    return rank, rank_label


def extract_sidebar_values(card):
    result = {}
    for heading in card.select("h4"):
        label = clean_text(" ".join(heading.stripped_strings))
        values = []
        sibling = heading
        while True:
            sibling = sibling.find_next_sibling()
            if sibling is None or sibling.name == "h4":
                break
            if sibling.name == "p":
                text = clean_text(" ".join(sibling.stripped_strings))
                if text:
                    values.append(text)
        result[label] = values
    return result


def clean_summary(card):
    for p in card.select("p"):
        text = clean_text(" ".join(p.stripped_strings))
        if not text or "Read More" not in text:
            continue
        text = re.sub("\\s*Read More\\s*\\u00bb?$", "", text).strip()
        return text or None
    return None


def parse_cards(html):
    soup = BeautifulSoup(html, "html.parser")
    cards = soup.select('[data-hook="card"]')
    rows = []

    for card in cards:
        link = card.select_one("a.card-name")
        name_node = link.select_one("h3") if link else card.select_one("h3")
        school_name = clean_text(" ".join(name_node.stripped_strings)) if name_node else None
        url = urljoin(URL, link.get("href")) if link and link.get("href") else None
        rank, rank_label = extract_rank_info(card)
        location = extract_location(card)
        city, state = split_city_state(location)
        stats = extract_sidebar_values(card)
        tuition_values = stats.get("TUITION AND FEES (MASTER'S)", [])
        enrollment_values = stats.get("ENROLLMENT (FULL-TIME)", [])

        if not school_name or not url:
            continue

        rows.append({
            "rank": rank,
            "rank_label": rank_label,
            "school_name": school_name,
            "city": city,
            "state": state,
            "tuition": " | ".join(tuition_values) if tuition_values else None,
            "enrollment_full_time": enrollment_values[0] if enrollment_values else None,
            "summary": clean_summary(card),
            "url": url,
        })

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("Tidak ada card universitas yang berhasil diparse dari HTML.")
    df = df.drop_duplicates(subset=["school_name", "url"], keep="first")
    df = df.sort_values(by=["rank", "school_name"], na_position="last").reset_index(drop=True)
    return df


html = fetch_html(URL)
if not RAW_HTML_PATH.exists():
    RAW_HTML_PATH.write_text(html, encoding="utf-8")

df = parse_cards(html)
if len(df) <= 10:
    print("Peringatan: hasil masih 10 atau kurang. Biasanya ini berarti HTML yang dipakai belum memuat hasil setelah tombol Load More.")
df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
df.to_excel(XLSX_PATH, index=False)

print(f"Jumlah universitas: {len(df)}")
print(f"HTML cache: {RAW_HTML_PATH.resolve()}")
print(f"CSV: {CSV_PATH.resolve()}")
print(f"XLSX: {XLSX_PATH.resolve()}")
df.head(20)


HTML berhasil diambil dengan selenium
Jumlah universitas: 205
HTML cache: C:\project-gabut\uni-scrapper\output\usnews_cs_raw.html
CSV: C:\project-gabut\uni-scrapper\output\usnews_cs_rankings.csv
XLSX: C:\project-gabut\uni-scrapper\output\usnews_cs_rankings.xlsx


,rank,rank_label,school_name,city,state,tuition,enrollment_full_time,summary,url
0,1,# 1 in Best Computer Science Schools (tie),Carnegie Mellon University,Pittsburgh,PA,"$61,406 per year (full-time) | $1,845 per cred...",1887,The School of Computer Science at Carnegie Mel...,https://www.usnews.com/best-graduate-schools/t...
1,1,# 1 in Best Computer Science Schools (tie),Massachusetts Institute of Technology,Cambridge,MA,"$62,396 per year (full-time)",831,The Department of Electrical Engineering and C...,https://www.usnews.com/best-graduate-schools/t...
2,1,# 1 in Best Computer Science Schools (tie),Stanford University,Stanford,CA,"$65,865 per year (full-time)",1004,The Department of Computer Science at Stanford...,https://www.usnews.com/best-graduate-schools/t...
3,4,# 4 in Best Computer Science Schools,University of California--Berkeley,Berkeley,CA,"$37,892 per year (full-time)",291,The Department of Electrical Engineering and C...,https://www.usnews.com/best-graduate-schools/t...
4,5,# 5 in Best Computer Science Schools,University of Illinois--Urbana-Champaign,Urbana,IL,"$43,670 per year (full-time)",2902,The Siebel School of Computing and Data Scienc...,https://www.usnews.com/best-graduate-schools/t...
5,6,# 6 in Best Computer Science Schools,Princeton University,None,NJ,N/A,N/A,None,https://www.usnews.com/best-graduate-schools/t...
6,7,# 7 in Best Computer Science Schools (tie),Cornell University,Ithaca,NY,"$75,384 per year (full-time)",535,The Department of Computer Science at Cornell ...,https://www.usnews.com/best-graduate-schools/t...
7,7,# 7 in Best Computer Science Schools (tie),Georgia Institute of Technology,None,GA,"$31,331 per year (full-time) | $2,009 per cred...",1329,The College of Computing at Georgia Institute ...,https://www.usnews.com/best-graduate-schools/t...
8,7,# 7 in Best Computer Science Schools (tie),University of Washington,Seattle,WA,"$1,393 per credit (part-time)",520,The Computer Science and Engineering at Univer...,https://www.usnews.com/best-graduate-schools/t...
9,10,# 10 in Best Computer Science Schools,University of Texas--Austin,Austin,TX,"$18,162 per year (full-time) | $1,913 per cred...",307,The Department of Computer Sciences at Univers...,https://www.usnews.com/best-graduate-schools/t...
